Convnext_contextUnet


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm


# ============================================================
# Context Decoder Block (không có Coordinate Attention)
# ============================================================

class ContextDecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_ch,
            out_ch,
            3,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_ch)

        self.conv2 = nn.Conv2d(
            out_ch,
            out_ch,
            3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_ch)

        self.relu = nn.ReLU(inplace=True)

        self.shortcut = (
            nn.Conv2d(in_ch, out_ch, 1, bias=False)
            if in_ch != out_ch
            else nn.Identity()
        )

    def forward(self, x):

        residual = self.shortcut(x)

        x = self.relu(self.bn1(self.conv1(x)))

        x = self.bn2(self.conv2(x))

        x = x + residual

        return self.relu(x)


# ============================================================
# Projection Adapter
# ============================================================

class Projection(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_ch,
                out_ch,
                3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


# ============================================================
# Proposed Model
# ============================================================

class ConvNeXtV2_ContextUNet(nn.Module):

    def __init__(
        self,
        model_name="convnextv2_pico.fcmae_ft_in1k",
        pretrained=True,
        out_channels=1
    ):
        super().__init__()

        self.register_buffer(
            "mean",
            torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
        )

        self.register_buffer(
            "std",
            torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)
        )

        self.encoder = timm.create_model(
            model_name,
            pretrained=pretrained,
            features_only=True,
            out_indices=(0,1,2,3)
        )

        enc_ch = self.encoder.feature_info.channels()

        print("Encoder channels:", enc_ch)

        self.proj1 = Projection(enc_ch[0], 64)
        self.proj2 = Projection(enc_ch[1], 128)
        self.proj3 = Projection(enc_ch[2], 256)
        self.proj4 = Projection(enc_ch[3], 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = ContextDecoderBlock(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = ContextDecoderBlock(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = ContextDecoderBlock(128, 64)

        self.final = nn.Sequential(
            ContextDecoderBlock(64, 64),
            nn.Conv2d(64, out_channels, 1)
        )

    def normalize(self, x):
        return (x - self.mean) / self.std

    def forward(self, x):
        input_size = x.shape[2:]

        x = self.normalize(x)

        e1, e2, e3, e4 = self.encoder(x)

        e1 = self.proj1(e1)
        e2 = self.proj2(e2)
        e3 = self.proj3(e3)
        e4 = self.proj4(e4)

        d3 = self.up3(e4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        out = self.final(d1)

        out = F.interpolate(out, size=input_size, mode='bilinear', align_corners=False)

        return out

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ConvNeXtV2_ContextUNet(
    model_name="convnextv2_pico.fcmae_ft_in1k",
    pretrained=False,
    out_channels=1
).to(device)

ckpt = torch.load("/content/best_convnextv2_unet_boundary_dice_crack500.pth",
                  map_location=device)

model.load_state_dict(ckpt, strict=False)
model.eval()

Encoder channels: [64, 128, 256, 512]


ConvNeXtV2_ContextUNet(
  (encoder): FeatureListNet(
    (stem_0): Conv2d(3, 64, kernel_size=(4, 4), stride=(4, 4))
    (stem_1): LayerNorm2d((64,), eps=1e-06, elementwise_affine=True)
    (stages_0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(64, 64, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=64)
          (norm): LayerNorm2d((64,), eps=1e-06, elementwise_affine=True)
          (mlp): GlobalResponseNormMlp(
            (fc1): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1))
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (grn): GlobalResponseNorm()
            (fc2): Conv2d(256, 64, kernel_size=(1, 1), stride=(1, 1))
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(64, 64, kernel_size=(7, 7), strid

U-net


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------- DoubleConv (dùng chung cho encoder và decoder) --------------------
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

# -------------------- Downsample (maxpool) --------------------
class Down(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.mp = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch, dropout)
    def forward(self, x):
        x = self.mp(x)
        x = self.conv(x)
        return x

# -------------------- Upsample (transpose conv) --------------------
class Up(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch, dropout)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        # Match size (in case of odd dimensions)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        x = self.conv(x)
        return x

# -------------------- U-Net thuần (không pretrained) --------------------
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=(64, 128, 256, 512), dropout=0.1):
        super().__init__()
        self.inc = DoubleConv(in_channels, features[0], dropout=dropout)

        self.down1 = Down(features[0], features[1], dropout=dropout)
        self.down2 = Down(features[1], features[2], dropout=dropout)
        self.down3 = Down(features[2], features[3], dropout=dropout)

        self.bottleneck = DoubleConv(features[3], features[3], dropout=dropout)

        self.up1 = Up(features[3], features[2], dropout=dropout)
        self.up2 = Up(features[2], features[1], dropout=dropout)
        self.up3 = Up(features[1], features[0], dropout=dropout)

        # Final up to full resolution (since original U-Net ends at same resolution)
        # However, features[0] is at H/4? Actually in original U-Net, after up3 we are at H/2? Let's clarify.
        # In standard U-Net: input HxW -> after inc -> HxW, after down1 -> H/2, down2 -> H/4, down3 -> H/8.
        # But our features[0] = 64, after down3 we are at H/8? Wait, let's re-evaluate.
        # Actually standard U-Net uses 4 downsampling levels to get to H/16, but here we have 3 down levels (features length 4 includes inc).
        # To keep consistent with earlier code (which expected e1 at H/4), we need to adjust.
        # Simpler: follow original U-Net with 4 downsampling steps.
        # Let's redesign to have 4 down steps so that final decoder output is at full resolution.
        # But user wants to replace ConvNeXtV2 encoder with standard CNN. I'll provide a common U-Net.
        # I'll implement classic U-Net with 4 down and 4 up (like original).
        pass

# Better to provide a clean standard U-Net that matches typical segmentation tasks.
# I'll rewrite properly:

class StandardUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512], dropout=0.1):
        super().__init__()
        # Encoder
        self.enc1 = DoubleConv(in_channels, features[0], dropout=dropout)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(features[0], features[1], dropout=dropout)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(features[1], features[2], dropout=dropout)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = DoubleConv(features[2], features[3], dropout=dropout)
        self.pool4 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(features[3], features[3]*2, dropout=dropout)

        # Decoder
        self.up4 = nn.ConvTranspose2d(features[3]*2, features[3], kernel_size=2, stride=2)
        self.dec4 = DoubleConv(features[3]*2, features[3], dropout=dropout)

        self.up3 = nn.ConvTranspose2d(features[3], features[2], kernel_size=2, stride=2)
        self.dec3 = DoubleConv(features[2]*2, features[2], dropout=dropout)

        self.up2 = nn.ConvTranspose2d(features[2], features[1], kernel_size=2, stride=2)
        self.dec2 = DoubleConv(features[1]*2, features[1], dropout=dropout)

        self.up1 = nn.ConvTranspose2d(features[1], features[0], kernel_size=2, stride=2)
        self.dec1 = DoubleConv(features[0]*2, features[0], dropout=dropout)

        # Final convolution to reduce to desired channels
        self.out_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)          # H x W
        p1 = self.pool1(e1)        # H/2 x W/2
        e2 = self.enc2(p1)         # H/2 x W/2
        p2 = self.pool2(e2)        # H/4 x W/4
        e3 = self.enc3(p2)         # H/4 x W/4
        p3 = self.pool3(e3)        # H/8 x W/8
        e4 = self.enc4(p3)         # H/8 x W/8
        p4 = self.pool4(e4)        # H/16 x W/16

        # Bottleneck
        b = self.bottleneck(p4)    # H/16 x W/16

        # Decoder
        d4 = self.up4(b)           # H/8 x W/8
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)         # H/8 x W/8

        d3 = self.up3(d4)          # H/4 x W/4
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)         # H/4 x W/4

        d2 = self.up2(d3)          # H/2 x W/2
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)         # H/2 x W/2

        d1 = self.up1(d2)          # H x W
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)         # H x W

        out = self.out_conv(d1)
        return out

# Example usage:
# model = StandardUNet(in_channels=3, out_channels=1, features=[64,128,256,512], dropout=0.1)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = StandardUNet(
    out_channels=1
).to(device)

ckpt = torch.load("/content/best_standard_unet_boundary_dice_crack500.pth",
                  map_location=device)

model.load_state_dict(ckpt, strict=False)
model.eval()

StandardUNet(
  (enc1): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Dropout2d(p=0.1, inplace=False)
      (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU(inplace=True)
    )
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (enc2): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Dropout2d(p=0.1, inplace=False)
      (4): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bi

convnext+unet decoder


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# -------------------- DoubleConv (chuẩn, không attention) --------------------
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

# -------------------- Projection (align channels) --------------------
class Projection(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

# -------------------- Base U-Net with ConvNeXtV2 encoder --------------------
class UNet(nn.Module):
    def __init__(
        self,
        model_name="convnextv2_pico.fcmae_ft_in1k",
        pretrained=True,
        out_channels=1,
        decoder_channels=(64, 128, 256, 512),
        dropout=0.1
    ):
        super().__init__()
        # ImageNet normalization
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1))

        # Encoder
        self.encoder = timm.create_model(
            model_name,
            pretrained=pretrained,
            features_only=True,
            out_indices=(0,1,2,3)
        )
        enc_ch = self.encoder.feature_info.channels()
        print(f"Encoder channels: {enc_ch}")

        # Projection
        f1, f2, f3, f4 = decoder_channels
        self.proj1 = Projection(enc_ch[0], f1)
        self.proj2 = Projection(enc_ch[1], f2)
        self.proj3 = Projection(enc_ch[2], f3)
        self.proj4 = Projection(enc_ch[3], f4)

        # Bottleneck
        self.bottleneck = DoubleConv(f4, f4, dropout=dropout)

        # Decoder
        self.up3 = nn.ConvTranspose2d(f4, f3, 2, stride=2)
        self.dec3 = DoubleConv(f3 + f3, f3, dropout=dropout)

        self.up2 = nn.ConvTranspose2d(f3, f2, 2, stride=2)
        self.dec2 = DoubleConv(f2 + f2, f2, dropout=dropout)

        self.up1 = nn.ConvTranspose2d(f2, f1, 2, stride=2)
        self.dec1 = DoubleConv(f1 + f1, f1, dropout=dropout)

        # Upsample từ stride4 lên full res (2x + 2x)
        self.up0_1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.up0_conv1 = DoubleConv(f1, 32, dropout=0.0)
        self.up0_2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.up0_conv2 = DoubleConv(32, 32, dropout=0.0)

        self.out_conv = nn.Conv2d(32, out_channels, 1)

        # Khởi tạo trọng số cho decoder/projection (encoder giữ pretrained)
        self._init_weights()

    def _init_weights(self):
        for name, m in self.named_modules():
            if not name.startswith("encoder"):
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                elif isinstance(m, nn.BatchNorm2d):
                    nn.init.constant_(m.weight, 1)
                    nn.init.constant_(m.bias, 0)

    def _match_size(self, x, ref):
        if x.shape[-2:] != ref.shape[-2:]:
            x = F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)
        return x

    def normalize(self, x):
        return (x - self.mean) / self.std

    def forward(self, x):
        x = self.normalize(x)

        # Encoder
        e1, e2, e3, e4 = self.encoder(x)   # strides 4,8,16,32

        # Projection
        e1 = self.proj1(e1)
        e2 = self.proj2(e2)
        e3 = self.proj3(e3)
        e4 = self.proj4(e4)

        # Bottleneck
        b = self.bottleneck(e4)

        # Decoder with skip
        d3 = self.up3(b)
        d3 = self._match_size(d3, e3)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self._match_size(d2, e2)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self._match_size(d1, e1)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        # Lên full resolution
        out = self.up0_1(d1)
        out = self.up0_conv1(out)
        out = self.up0_2(out)
        out = self.up0_conv2(out)
        out = self.out_conv(out)

        return out

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = UNet(
    out_channels=1
).to(device)

ckpt = torch.load("/content/best_convnextv2basse_unet_boundary_dice_crack500.pth",
                  map_location=device)

model.load_state_dict(ckpt, strict=False)
model.eval()


Encoder channels: [64, 128, 256, 512]


UNet(
  (encoder): FeatureListNet(
    (stem_0): Conv2d(3, 64, kernel_size=(4, 4), stride=(4, 4))
    (stem_1): LayerNorm2d((64,), eps=1e-06, elementwise_affine=True)
    (stages_0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(64, 64, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=64)
          (norm): LayerNorm2d((64,), eps=1e-06, elementwise_affine=True)
          (mlp): GlobalResponseNormMlp(
            (fc1): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1))
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (grn): GlobalResponseNorm()
            (fc2): Conv2d(256, 64, kernel_size=(1, 1), stride=(1, 1))
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(64, 64, kernel_size=(7, 7), stride=(1, 1), padding=

In [ ]:
!pip install thop

In [ ]:
import time
import torch
from thop import profile

def measure_performance(model, model_name, device, input_size=(1, 3, 384, 384)):
    print(f"\n--- Measuring: {model_name} ---")
    model.eval()
    inputs = torch.randn(input_size).to(device)

    # 1. Tính FLOPs và Parameters
    flops, params = profile(model, inputs=(inputs, ), verbose=False)
    print(f"Input Size: {input_size}")
    print(f"FLOPs: {flops / 1e9:.2f} GFLOPs")
    print(f"Parameters: {params / 1e6:.2f} M")

    # 2. Tính Model Size
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    size_all_mb = (param_size + buffer_size) / 1024**2
    print(f"Model Size: {size_all_mb:.2f} MB")

    # 3. Đo Inference Time
    # Warm up
    for _ in range(10):
        _ = model(inputs)

    if device == 'cuda':
        torch.cuda.synchronize()

    start_time = time.time()
    num_iterations = 100
    with torch.no_grad():
        for _ in range(num_iterations):
            _ = model(inputs)
            if device == 'cuda':
                torch.cuda.synchronize()

    end_time = time.time()
    avg_time = (end_time - start_time) / num_iterations
    print(f"Inference time: {avg_time * 1000:.2f} ms per image")

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Khởi tạo lại model ConvNeXtV2_ContextUNet
model_convnext = ConvNeXtV2_ContextUNet(pretrained=False).to(device)
measure_performance(model_convnext, "ConvNeXtV2_ContextUNet", device)

# 2. Khởi tạo lại model StandardUNet
model_standard = StandardUNet(out_channels=1).to(device)
measure_performance(model_standard, "StandardUNet", device)

# 3. Khởi tạo lại model UNet (ConvNeXtV2 encoder)
model_unet = UNet(pretrained=False).to(device)
measure_performance(model_unet, "UNet (ConvNeXtV2 Encoder)", device)


Encoder channels: [64, 128, 256, 512]

--- Measuring: ConvNeXtV2_ContextUNet ---
Input Size: (1, 3, 384, 384)
FLOPs: 10.26 GFLOPs
Parameters: 14.92 M
Model Size: 57.03 MB
Inference time: 9.33 ms per image

--- Measuring: StandardUNet ---
Input Size: (1, 3, 384, 384)
FLOPs: 123.16 GFLOPs
Parameters: 31.04 M
Model Size: 118.44 MB
Inference time: 15.39 ms per image
Encoder channels: [64, 128, 256, 512]

--- Measuring: UNet (ConvNeXtV2 Encoder) ---
Input Size: (1, 3, 384, 384)
FLOPs: 13.90 GFLOPs
Parameters: 19.44 M
Model Size: 74.29 MB
Inference time: 10.11 ms per image


In [ ]:
import time
import torch

def measure_fps(model, model_name, device, input_size=(1, 3, 384, 384)):
    model.eval()
    inputs = torch.randn(input_size).to(device)

    # Warm up để GPU tối ưu hoá tính toán
    for _ in range(10):
        _ = model(inputs)

    if device == 'cuda':
        torch.cuda.synchronize()

    start_time = time.time()
    num_iterations = 100

    with torch.no_grad():
        for _ in range(num_iterations):
            _ = model(inputs)
            if device == 'cuda':
                torch.cuda.synchronize()

    end_time = time.time()
    avg_time = (end_time - start_time) / num_iterations
    fps = 1.0 / avg_time

    print(f"{model_name: <30} | FPS: {fps:.2f} frames/sec | Avg Time: {avg_time * 1000:.2f} ms")

print("--- Đo lường FPS cho các mô hình ---")
measure_fps(model_convnext, "ConvNeXtV2_ContextUNet", device)
measure_fps(model_standard, "StandardUNet", device)
measure_fps(model_unet, "UNet (ConvNeXtV2 Encoder)", device)


--- Đo lường FPS cho các mô hình ---
ConvNeXtV2_ContextUNet         | FPS: 107.42 frames/sec | Avg Time: 9.31 ms
StandardUNet                   | FPS: 66.21 frames/sec | Avg Time: 15.10 ms
UNet (ConvNeXtV2 Encoder)      | FPS: 98.76 frames/sec | Avg Time: 10.13 ms


In [6]:
pip install gradio -q

In [9]:
import os
import cv2
import time
import tempfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import gradio as gr

# =========================
# 1. MODEL DEFINITION
# =========================

class ContextDecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        residual = self.shortcut(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = x + residual
        return self.relu(x)

class Projection(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class ConvNeXtV2_ContextUNet(nn.Module):
    def __init__(self, model_name="convnextv2_pico.fcmae_ft_in1k", pretrained=False, out_channels=1):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.encoder = timm.create_model(model_name, pretrained=pretrained, features_only=True, out_indices=(0, 1, 2, 3))
        enc_ch = self.encoder.feature_info.channels()
        self.proj1 = Projection(enc_ch[0], 64)
        self.proj2 = Projection(enc_ch[1], 128)
        self.proj3 = Projection(enc_ch[2], 256)
        self.proj4 = Projection(enc_ch[3], 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = ContextDecoderBlock(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = ContextDecoderBlock(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = ContextDecoderBlock(128, 64)
        self.final = nn.Sequential(ContextDecoderBlock(64, 64), nn.Conv2d(64, out_channels, 1))

    def normalize(self, x): return (x - self.mean) / self.std

    def forward(self, x):
        input_size = x.shape[2:]
        x = self.normalize(x)
        e1, e2, e3, e4 = self.encoder(x)
        e1, e2, e3, e4 = self.proj1(e1), self.proj2(e2), self.proj3(e3), self.proj4(e4)
        d3 = self.dec3(torch.cat([self.up3(e4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        out = F.interpolate(self.final(d1), size=input_size, mode="bilinear", align_corners=False)
        return out

# =========================
# 2. LOAD MODEL
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WEIGHT_PATH = "/content/best_convnextv2_unet_boundary_dice_crack500.pth"
model = ConvNeXtV2_ContextUNet().to(DEVICE)
if os.path.exists(WEIGHT_PATH):
    ckpt = torch.load(WEIGHT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt)
    print("Model loaded.")
model.eval()

# =========================
# 3. UTILS
# =========================
@torch.no_grad()
def predict_mask(frame_bgr, threshold=0.4, input_size=384):
    h, w = frame_bgr.shape[:2]
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (input_size, input_size))
    x = torch.from_numpy(resized).float().permute(2, 0, 1).unsqueeze(0).to(DEVICE) / 255.0
    prob = torch.sigmoid(model(x))[0, 0].cpu().numpy()
    mask = (prob > threshold).astype(np.uint8) * 255
    return cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

def process_video(video_path, threshold, alpha, input_size):
    if not video_path: return None, "No video."
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w, h = int(cap.get(3)), int(cap.get(4))
    out_path = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False).name
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    start_time = time.time()
    frame_count = 0
    while True:
        t1 = time.time()
        ret, frame = cap.read()
        if not ret: break

        mask = predict_mask(frame, threshold, input_size)
        overlay = frame.copy()
        overlay[mask > 0] = (0, 0, 255)
        output = cv2.addWeighted(frame, 1 - alpha, overlay, alpha, 0)

        frame_count += 1
        curr_fps = 1.0 / (time.time() - t1)
        cv2.putText(output, f"Processing FPS: {curr_fps:.1f}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        writer.write(output)

    cap.release(); writer.release()
    total_time = time.time() - start_time
    return out_path, f"Processed {frame_count} frames in {total_time:.1f}s (Avg FPS: {frame_count/total_time:.1f})"

# =========================
# 4. UI
# =========================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 Crack AI Dashboard with Real-time FPS Display")
    with gr.Row():
        v_in = gr.Video(label="Input Video")
        v_out = gr.Video(label="Output Video (with FPS)")
    with gr.Row():
        t_sld = gr.Slider(0.1, 0.9, 0.4, step=0.05, label="Threshold")
        a_sld = gr.Slider(0.1, 0.9, 0.45, step=0.05, label="Alpha")
        s_drp = gr.Dropdown([256, 384, 512], value=384, label="Input Size")
    btn = gr.Button("Run Pipeline", variant="primary")
    info = gr.Textbox(label="System Info")
    btn.click(process_video, [v_in, t_sld, a_sld, s_drp], [v_out, info])

demo.launch(share=True, debug=True)

Model loaded.


/tmp/ipykernel_9582/3770911430.py:133: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://45b96f40654601330c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 